# 02a — Model A

**Purpose.** Develop one model end to end: its own preprocessing, feature
engineering, tuning and training, evaluated on the validation split.

**Inputs.** `data/processed/train.parquet` — **the train split only**.

**Outputs.** `models/model_a.pkl`, containing the fitted preprocessor *and* the
fitted model, plus MLflow runs.

---

### Copy this notebook per model

`cp notebooks/02a_model_a.ipynb notebooks/02b_<your_model>.ipynb`, alongside a
new module in `src/models/` and a new config in `configs/models/`. The number
already means "model", so no `model_` prefix in the filename.

### What belongs here

Everything that **learns from data**: imputation, statistical outlier handling,
scaling, encoding, feature engineering and selection, architecture, tuning,
training. All fitted on the training split only.

### What does not

- **Splitting** — call `src.data.split`, so every model sees the same folds.
- **Metric definitions** — call `src.evaluation.metrics`, so the results table
  compares models rather than measurement choices.
- **The test split** — it stays frozen until notebook 03.

## 1. Setup

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from src.config import load_config
from src.utils.seed import set_seed

cfg = load_config(overrides=["models=model_a"])
set_seed(cfg.seed)

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the training split

⚠️ **Only `"train"`.** Reading the test split here — even just to look — is how
a held-out estimate quietly stops being held out.

In [ ]:
from src.data.load import load_processed

train_full = load_processed(cfg, "train")
print(f"{len(train_full):,} training records")
train_full.head()

## 3. Train/validation split

From the shared splitter, using `split.val_size` in `configs/data.yaml`. Do not
introduce a validation fraction in the model config: if each model tuned and
early-stopped against different data, notebook 03 would be comparing folds.

In [ ]:
from src.data.split import assert_no_group_leakage, train_val_split

train_df, val_df = train_val_split(train_full, cfg)
assert_no_group_leakage(train_df, val_df, cfg)

target = cfg.target
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_val, y_val = val_df.drop(columns=[target]), val_df[target]
print(f"train: {len(X_train):,}   val: {len(X_val):,}")

## 4. Model-specific preprocessing

Fitted on `X_train` only, then applied to `X_val` with `transform`. Fitting on
train+val inflates the validation score and hides overfitting, which defeats the
purpose of having a validation split.

Build the pipeline in `src/models/model_a.py` (`build_preprocessor`) rather than
in a cell, so that training, evaluation and serving all use the same code.

In [ ]:
# Explore options here, then move the final pipeline into
# src/models/model_a.py::build_preprocessor.
#
# TODO: imputation strategy per column (and an indicator for informative gaps)
# TODO: statistical outlier handling — thresholds computed from X_train ONLY
# TODO: scaling for numeric columns, encoding for categoricals

## 5. Feature engineering and selection

Model-specific by design: transforms live with the model that needs them, not
in a shared `features/` package. Derive features from `X_train` statistics only.

In [ ]:
# TODO: construct candidate features
# TODO: select with a train-only criterion (importance, mutual information)
# TODO: fold the winners into build_preprocessor so they are serialised with
#       the model and applied identically at serving time

## 6. Define the model

Instantiated from `configs/models/model_a.yaml` through Hydra's `_target_`, so
the notebook and `task train` build exactly the same object.

In [ ]:
from hydra.utils import instantiate

model = instantiate(cfg.models)
model

## 7. Hyperparameter tuning

Scored on the validation split, with `src.evaluation.metrics`. Use the shared
`cv_folds` helper if you tune with cross-validation, so every model that does
uses the same folds.

Record the winning values in `configs/models/model_a.yaml` — a hyperparameter
that only exists in a notebook cell is lost the moment the kernel restarts.

In [ ]:
# TODO: define the search space
# TODO: search (grid / random / optuna), scoring with metrics.evaluate
# TODO: copy the best parameters into configs/models/model_a.yaml

## 8. Train the final model

With tracking. Log the shared metrics explicitly rather than relying on
`autolog`: framework autologging records different things for a boosted tree
than for a neural network, which is exactly the comparison this project
protects.

In [ ]:
from src.utils import tracking

with tracking.start_run(cfg, run_name="model_a"):
    tracking.log_config(cfg)
    model.fit(X_train, y_train, X_val, y_val)
    tracking.log_model_summary(model)

## 9. Validation analysis

How good, and — more usefully — where it is wrong.

In [ ]:
from src.evaluation.metrics import evaluate, evaluate_by_group

val_pred = model.predict(X_val)
scores = evaluate(y_val.to_numpy(), val_pred)
scores

In [ ]:
from src.evaluation.analysis import error_table, feature_importance

error_table(y_val.to_numpy(), val_pred, features=X_val, top_n=20)

In [ ]:
feature_importance(model, X_val, y_val.to_numpy())

# TODO: per-group breakdown with evaluate_by_group — an aggregate score hides
#       which slice the model fails on, and that is usually what matters.

## 10. Persist the artifact

The fitted preprocessor and the model are saved **together**. Notebook 03 and
the serving layer then only ever call `transform`, which is what prevents
train/serve skew.

In [ ]:
model.save(cfg.models.artifact_path)
print(f"wrote {cfg.models.artifact_path}")

# Version the artifact against this commit:
#   dvc add models/model_a.pkl && git add models/model_a.pkl.dvc

## 11. Handoff checklist

- [ ] The test split was never read in this notebook
- [ ] Every fitted transform saw `X_train` only
- [ ] The validation split came from `src.data.split`, with `val_size` from
      `configs/data.yaml`
- [ ] Scores came from `src.evaluation.metrics`
- [ ] Final hyperparameters are written back to `configs/models/model_a.yaml`
- [ ] `fit` / `predict` / `save` / `load` all work on the class
- [ ] The artifact contains the preprocessor **and** the model
- [ ] The run is recorded in MLflow